# Advanced Context-Aware Beverage Recommender

This notebook demonstrates SOTA techniques for context-aware recommendation:

1. **SASRec** (Sequence-Aware Session-based Recommender) - Transformer-based sequential recommendation
2. **Multi-Modal Fusion** - Combining drink content with weather, time, and occasion signals
3. **LinUCB Bandit** - Exploration-exploitation balancing
4. **Hybrid Ensemble** - Combining all components for robust recommendations

## Problem Statement

Personalized beverage recommendation based on multiple contextual signals:
- User's historical preferences (sequential modeling)
- Current weather conditions
- Time of day and day of week
- Social occasion (casual, celebration, pairing, etc.)
- Taste preferences (bitterness, sweetness, strength)

## Questions from Specification

**a. User base size**: We use synthetic data with ~500 users and ~10,000 interactions for demonstration.

**b. Explicit vs implicit feedback**: Using implicit feedback (views, ratings 1-5) which is standard for recommendation systems.

**c. Cold-start handling**: Hybrid content-based + popularity fallback for new drinks with no interaction history.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path(__file__).parent.parent))

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Data Loading and Exploration

In [ ]:
from models.data_layer import (
    generate_drink_catalog,
    generate_context_scenarios,
    generate_interaction_logs,
    get_drink_stats,
    get_context_stats
)

# Generate synthetic data for demonstration
print("Generating synthetic dataset...")
drinks_df = generate_drink_catalog(n_drinks=120, seed=42)
scenarios_df = generate_context_scenarios(n_scenarios=50, seed=42)
interactions_df = generate_interaction_logs(
    n_interactions=10000,
    n_users=500,
    drinks_df=drinks_df,
    scenarios_df=scenarios_df,
    seed=42
)

print("\n=== Drink Catalog Statistics ===")
stats = get_drink_stats(drinks_df)
for key, value in stats.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Display sample drinks
print("\n=== Sample Drinks ===")
display(drinks_df.head(10))

print("\n=== Sample Interactions ===")
display(interactions_df.head(10))

## 2. SASRec (Sequence-Aware Session-based Recommender)

In [ ]:
from models.sasrec import SASRecModel, build_sessions_from_interactions

print("=== Building Sessions from Interactions ===")
sessions, targets, item_to_idx = build_sessions_from_interactions(
    interactions_df,
    min_session_len=2,
    max_session_len=10
)

print(f"Number of sessions: {len(sessions)}")
print(f"Average session length: {sessions.shape[1]:.2f}")
print(f"Number of unique items: {len(item_to_idx)}")

In [ ]:
print("=== Training SASRec Model ===")
sasrec = SASRecModel(
    n_items=len(item_to_idx),
    d_model=64,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    max_seq_len=10,
    dropout_rate=0.1,
    lr=0.001,
    seed=42
)

history = sasrec.fit(
    sessions,
    targets,
    n_epochs=5,
    batch_size=32,
    val_split=0.2,
    early_stop_patience=3
)

In [ ]:
print("=== SASRec Recommendations ===")
sample_session_array = sessions[0]
recommendations = sasrec.get_recommendations(
    sample_session_array,
    excluded_items=None,
    top_k=5
)

print(f"\nSample session: {sample_session_array}")
print("Top 5 recommendations:")
for i, (item_idx, score) in enumerate(recommendations, 1):
    print(f"  {i}. Item {item_idx}: score={score:.4f}")

## 3. Multi-Modal Fusion Model

In [ ]:
from models.fusion import MultiModalFusionModel

print("=== Training Fusion Model ===")
fusion = MultiModalFusionModel(
    n_drinks=len(drinks_df),
    d_model=64,
    d_context=32,
    dropout_rate=0.1,
    lr=0.001,
    seed=42
)

fusion_history = fusion.fit(
    drinks_df,
    interactions_df,
    n_epochs=5,
    batch_size=32,
    val_split=0.2
)

In [ ]:
print("=== Fusion Recommendations for Different Contexts ===")
contexts = [
    {"weather": "sunny", "time_period": "afternoon", "occasion": "casual"},
    {"weather": "rainy", "time_period": "evening", "occasion": "celebration"},
    {"weather": "cold", "time_period": "morning", "occasion": "recovery"},
]

for ctx in contexts:
    print(f"\nContext: {ctx['weather']} weather, {ctx['time_period']}, {ctx['occasion']}")
    recs = fusion.get_recommendations(
        drinks_df,
        ctx['weather'],
        ctx['time_period'],
        ctx['occasion'],
        top_k=3
    )
    for i, (drink_id, score) in enumerate(recs, 1):
        drink = drinks_df[drinks_df['drink_id'] == drink_id].iloc[0]
        print(f"  {i}. {drink['name']} ({drink['type']}): score={score:.4f}")

## 4. LinUCB Bandit for Exploration-Exploitation

In [ ]:
from models.linucb import LinUCBRecommender

print("=== Training LinUCB from Interactions ===")
linucb = LinUCBRecommender(
    drink_ids=drinks_df['drink_id'].tolist(),
    d_features=16,
    alpha=1.0,
    reg_param=1.0,
    seed=42
)

linucb.train_from_interactions(
    interactions_df,
    d_features=16,
    n_epochs=5
)

In [ ]:
print("=== LinUCB Recommendations with Exploration Bonus ===")
recs = linucb.get_recommendations_with_exploration(
    user_id="U001",
    weather="sunny",
    time_period="afternoon",
    occasion="casual",
    bitterness_pref=0.5,
    sweetness_pref=0.5,
    strength_pref=0.5,
    top_k=5
)

for i, rec in enumerate(recs, 1):
    drink = drinks_df[drinks_df['drink_id'] == rec['drink_id']].iloc[0]
    print(f"{i}. {drink['name']}")
    print(f"   Expected reward: {rec['expected_reward']:.4f}")
    print(f"   Uncertainty: {rec['uncertainty']:.4f}")

## 5. Hybrid Recommender Engine

In [ ]:
from models.recommender import HybridRecommenderEngine

print("=== Training Hybrid Recommender ===")
hybrid = HybridRecommenderEngine(
    drink_df=drinks_df,
    interactions_df=interactions_df,
    d_model=64,
    alpha=1.0,
    sasrec_weights=0.3,
    fusion_weights=0.5,
    linucb_weights=0.2,
    seed=42
)

In [ ]:
print("=== Hybrid Recommendations ===")
test_cases = [
    {"user_id": "U001", "weather": "sunny", "time_period": "afternoon", "occasion": "casual"},
    {"user_id": "U002", "weather": "rainy", "time_period": "evening", "occasion": "celebration"},
]

for test_case in test_cases:
    print(f"\nContext: {test_case['weather']}, {test_case['time_period']}, {test_case['occasion']}")
    recs = hybrid.recommend(
        user_id=test_case['user_id'],
        weather=test_case['weather'],
        time_period=test_case['time_period'],
        occasion=test_case['occasion'],
        top_k=5
    )
    for j, rec in enumerate(recs, 1):
        print(f"  {j}. {rec['name']}: score={rec['overall_score']:.4f}")

## 6. Interactive Demo

In [ ]:
print("=== Interactive Recommendation Demo ===")
demo_contexts = [
    ("Summer Beach Day", "sunny", "afternoon", "social", 0.4, 0.3, 0.5),
    ("Cozy Rainy Evening", "rainy", "evening", "casual", 0.6, 0.4, 0.3),
    ("Business Meeting", "cloudy", "morning", "business", 0.5, 0.5, 0.5),
]

for name, weather, time_period, occasion, bitter, sweet, strength in demo_contexts:
    print(f"\n{'='*60}")
    print(f"Scenario: {name}")
    print(f"Context: {weather}, {time_period}, {occasion}")
    
    recs = hybrid.recommend(
        user_id="U001",
        weather=weather,
        time_period=time_period,
        occasion=occasion,
        bitterness_pref=bitter,
        sweetness_pref=sweet,
        strength_pref=strength,
        top_k=3
    )
    
    for i, rec in enumerate(recs, 1):
        drink = drinks_df[drinks_df['drink_id'] == rec['drink_id']].iloc[0]
        print(f"\n{i}. {drink['name']}")
        print(f"   Type: {drink['type']} | ABV: {drink['abv']}%")
        print(f"   Score: {rec['overall_score']:.4f}")
        
        context = {
            "weather": weather,
            "time_period": time_period,
            "occasion": occasion
        }
        explanation = hybrid.explain_recommendation(rec['drink_id'], context)
        print(f"   Why: {explanation}")

## Summary

This notebook demonstrated a complete SOTA context-aware beverage recommender:

### Key Components
1. **SASRec** (30%): Transformer-based sequential recommendation
2. **Multi-Modal Fusion** (50%): Context-aware content fusion
3. **LinUCB** (20%): Exploration-exploitation balancing

### Results
- **Dataset**: 120 drinks, 10,000 interactions, 500 users
- **Context-aware**: Adapts to weather, time, occasion
- **Personalized**: Learns from user history
- **Diverse**: LinUCB ensures exploration